# RedZone Teacher 32B blind re-grade on the REAL-outcome label (EXP-1403; NBA, then NFL; self-contained)

Nick, Nov 5 1:50 AM: the old lock label was our own clock rule, and 150 blind answers were too few. This asks the 32B
Teacher the blind question on 700 TRAIN moments per league (NBA, NFL) and grades it and the fast head on lane 3's
real-outcome label y_out (EXP-302, #278: a lead change or catch, or a win-probability swing, in the next few minutes),
with balanced accuracy and 95% night-bootstrap CIs. **Bar:** the Teacher is worth distilling in a league only if its
CI lower bound beats the head's point estimate, with at least 600 answered (fewer = PENDING).

Plan B moments (ESPN), **personal-use, not for commercial release**. No Drive, no secrets. Code comes from three
public commits: the kaggle-inputs code pack, #278's label, and this branch's re-grade script.

Runtime: **A100 GPU**, then Runtime > **Run all**. About 2.7 h, **~35 CU** (moment build 1 h, Teacher ~1.5 h for
1,400 blind asks, grading a few minutes). If the runtime restarts but keeps its disk, Run all again: moments and
answers are kept. The last cell prints grades.json and downloads blind_regrade_out.zip (hand it to crew E).

In [ ]:
KPIN = "eac55da9b50f346a39dc8df393b7dc568fcd6bc7"   # kaggle-inputs: code pack (moment builder, run_teacher.py)
LPIN = "89fb5471ffb306e6ab6c438c044098a97f7790a0"   # #278 (EXP-302): redzone/lab/outcome_label.py + reasoner_data.py
EPIN = "2d3c3eae20e4668fb3984626b9b801dd8bce6c2c"   # crew E: agent/kaggle/blind_regrade.py
REPO = "BouncySlime1215/redzone-everything"
DATA_PLAN = "B"                 # Nick 11:20 PM Nov 4: personal use only
LEAGUES = "nba nfl"             # Nick 2:00 AM Nov 5: NBA first, then NFL
PER_LEAGUE = 700                # asked per league; >= 600 must come back valid
BUILD_H = 1.0
TEACHER_H = 1.6
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}

In [ ]:
import json, os, shutil, subprocess, sys, tarfile, time, urllib.request, zipfile
T_START = time.time()
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)
RUN = "/content/rzblind"
os.makedirs(RUN, exist_ok=True)

def fetch(ref, dst, keep=lambda p: True):
    if os.path.exists(f"{dst}/.done"):
        return
    os.makedirs(dst, exist_ok=True)
    for i in range(4):
        try:
            urllib.request.urlretrieve(f"https://codeload.github.com/{REPO}/tar.gz/{ref}", f"{RUN}/x.tar.gz")
            break
        except Exception as e:
            print(" retry", type(e).__name__, e); time.sleep(20 * (i + 1))
    else:
        raise SystemExit("could not download the code; Run all again in a minute")
    with tarfile.open(f"{RUN}/x.tar.gz") as t:
        for m in t.getmembers():
            parts = m.name.split("/", 1)
            if len(parts) == 2 and parts[1] and ".." not in parts[1].split("/") and keep(parts[1]):
                m.name = parts[1]
                t.extract(m, dst, filter="data")
    os.remove(f"{RUN}/x.tar.gz")
    open(f"{dst}/.done", "w").close()

PACK = f"{RUN}/kaggle-inputs"
fetch(KPIN, PACK, keep=lambda p: p.endswith("rz_code.tar.gz") or p.startswith("data/reasoner"))
CODE = f"{RUN}/rzcode/code"
if not os.path.exists(f"{CODE}/agent/kaggle/run_teacher.py"):
    with tarfile.open(f"{PACK}/rz_code.tar.gz") as t:
        t.extractall(f"{RUN}/rzcode", filter="data")
LBL = f"{RUN}/label"
fetch(LPIN, LBL, keep=lambda p: p in ("redzone/lab/outcome_label.py", "redzone/lab/reasoner_data.py"))
for f in ("outcome_label.py", "reasoner_data.py"):                # overlay #278's label onto the code pack
    shutil.copy(f"{LBL}/redzone/lab/{f}", f"{CODE}/redzone/lab/{f}")
bm = f"{CODE}/agent/kaggle/build_moments_kaggle.py"               # carry y_out onto every moment row
src = open(bm).read()
old = '"target": tgt, **{k: r[k] for k in ("y", "big_next"'
if old in src:
    open(bm, "w").write(src.replace(old, '"target": tgt, **{k: r[k] for k in ("y", "y_out", "flip", "wp_swing", "close_finish", "big_next"'))
elif '"y_out"' not in src:
    raise SystemExit("build_moments_kaggle.py changed shape: cannot add y_out (tell crew E)")
E = f"{RUN}/data_e"
fetch(EPIN, E, keep=lambda p: p == "agent/kaggle/blind_regrade.py")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "httpx", "pyyaml", "lightgbm"], check=False)
print("code ready")

In [ ]:
env = {**os.environ, "RZ_DATA_PLAN": DATA_PLAN, "RZ_CODE_PACK": f"kaggle-inputs@{KPIN}+label@{LPIN[:7]}+e@{EPIN[:7]}",
       "PYTHONUNBUFFERED": "1", "PYTHONPATH": CODE}
OUT = f"{RUN}/out"; MOM = f"{OUT}/moments"
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, CODE)
from agent.kaggle.teacher_only import gate
gate(DATA_PLAN, "espn")          # ESPN moments are plan B only
if not os.path.exists(f"{MOM}/DONE"):
    subprocess.run([sys.executable, "-u", "agent/kaggle/build_moments_kaggle.py", "--out", MOM, "--leagues",
                    *LEAGUES.split(), "--budget-h", str(BUILD_H)], cwd=CODE, env=env, check=True)

In [ ]:
log_path = f"{RUN}/blind_{time.strftime('%Y%m%d-%H%M')}.log"
with open(log_path, "w") as log:
    p = subprocess.Popen([sys.executable, "-u", f"{E}/agent/kaggle/blind_regrade.py", "--moments", MOM, "--out", OUT,
                          "--leagues", *LEAGUES.split(), "--per-league", str(PER_LEAGUE), "--budget-h", str(TEACHER_H),
                          "--system", f"{PACK}/data/reasoner/system.txt", "--code", CODE],
                         cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); log.write(line)
    print("exit code:", p.wait())

In [ ]:
wall_h = (time.time() - T_START) / 3600
gp = f"{OUT}/grades.json"
if os.path.exists(gp):
    g = json.load(open(gp))
    g["colab"] = {"gpu": GPU, "wall_h": round(wall_h, 2), "cu_spent": round(wall_h * RATE, 1) if RATE else None}
    json.dump(g, open(gp, "w"), indent=1)
    for lg, v in g["per_league"].items():
        t, h = v["teacher"], v["head"]
        print(f"{lg}: Teacher bal-acc {t['balanced_acc']:.3f} CI {t['ci95']} n={t['n']} | head {h['balanced_acc']:.3f} -> {v['verdict']}")
zp = f"{RUN}/blind_regrade_out.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("grades.json", "teacher_out.jsonl", "teacher_report.json", "blind_requests.jsonl"):
        if os.path.exists(f"{OUT}/{f}"):
            z.write(f"{OUT}/{f}", f)
    z.write(log_path, os.path.basename(log_path))
print(f"compute: {wall_h:.2f} h on {GPU}; zip {zp}")
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print("blind_regrade_out.zip is at", zp, f"({type(e).__name__})")